In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS cdc_lab;

USE SCHEMA cdc_lab;

In [0]:
BASE_PATH = "/Volumes/workspace/cdc_lab/cdc_files"

SNAPSHOT_01 = f"{BASE_PATH}/customers_snapshot_01.csv"
SNAPSHOT_02 = f"{BASE_PATH}/customers_snapshot_02.csv"
CDC_EVENTS = f"{BASE_PATH}/customers_cdc_events.csv"
CDC_EVENTS_REPLAY = f"{BASE_PATH}/customers_cdc_events_with_replay.csv"

### Read the initial dataset

In [0]:
df_initial = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SNAPSHOT_01)
)

display(df_initial)

customer_id,customer_name,city,email,updated_at
101,Aarav,Pune,aarav@example.com,2026-01-01
102,Diya,Mumbai,diya@example.com,2026-01-02
103,Kabir,Nashik,kabir@example.com,2026-01-03
104,Meera,Nagpur,meera@example.com,2026-01-04
105,Ishaan,Pune,ishaan@example.com,2026-01-05
106,Sara,Delhi,sara@example.com,2026-01-06
107,Vivaan,Mumbai,vivaan@example.com,2026-01-07
108,Anaya,Nanded,anaya@example.com,2026-01-08


**Check the schema:**

In [0]:
df_initial.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- updated_at: date (nullable = true)



### Method 1: CDC using Full Snapshot Comparison
**This method is used when the source system does not provide a CDC feed.**

- Source sends a complete customer snapshot every night.
- Databricks receives today's snapshot.
- We compare it with yesterday's snapshot.
- We identify inserted, updated and deleted records.
This is also called snapshot-based CDC.

**Step 1: Load the initial snapshot into Delta Lake**

In [0]:
df_initial.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cdc_lab.customers_batch_current")

In [0]:
%sql
SELECT *
FROM cdc_lab.customers_batch_current;

customer_id,customer_name,city,email,updated_at
101,Aarav,Pune,aarav@example.com,2026-01-01
102,Diya,Mumbai,diya@example.com,2026-01-02
103,Kabir,Nashik,kabir@example.com,2026-01-03
104,Meera,Nagpur,meera@example.com,2026-01-04
105,Ishaan,Pune,ishaan@example.com,2026-01-05
106,Sara,Delhi,sara@example.com,2026-01-06
107,Vivaan,Mumbai,vivaan@example.com,2026-01-07
108,Anaya,Nanded,anaya@example.com,2026-01-08


**Step 2: Read the next snapshot**

In [0]:
df_new = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SNAPSHOT_02)
)

### Step 3: Identify INSERT, UPDATE and DELETE

**We'll use a Full Outer Join.**

**Why?**

**Because a full outer join allows us to identify records present in either snapshot.**

In [0]:
from pyspark.sql.functions import col, when

old_df = df_initial.alias("old")
new_df = df_new.alias("new")

joined_df = old_df.join(
    new_df,
    col("old.customer_id") == col("new.customer_id"),
    "full_outer"
)

Now identify the changes:

In [0]:
changes_df = joined_df.select(

    col("new.customer_id").alias("customer_id"),

    when(
        col("old.customer_id").isNull(),
        "INSERT"
    )
    .when(
        col("new.customer_id").isNull(),
        "DELETE"
    )
    .when(
        ~(
            col("old.customer_name").eqNullSafe(col("new.customer_name")) &
            col("old.city").eqNullSafe(col("new.city")) &
            col("old.email").eqNullSafe(col("new.email"))
        ),
        "UPDATE"
    )
    .otherwise("NO_CHANGE").alias("operation"),

    col("new.customer_name").alias("customer_name"),
    col("new.city").alias("city"),
    col("new.email").alias("email"),
    col("new.updated_at").alias("updated_at")
)

In [0]:
display(changes_df)

customer_id,operation,customer_name,city,email,updated_at
105,UPDATE,Ishaan,Pune,ishaan.k@example.com,2026-02-02
104,NO_CHANGE,Meera,Nagpur,meera@example.com,2026-01-04
108,NO_CHANGE,Anaya,Nanded,anaya@example.com,2026-01-08
109,INSERT,Rohan,Aurangabad,rohan@example.com,2026-02-03
106,NO_CHANGE,Sara,Delhi,sara@example.com,2026-01-06
107,NO_CHANGE,Vivaan,Mumbai,vivaan@example.com,2026-01-07
102,UPDATE,Diya,Thane,diya@example.com,2026-02-01
101,NO_CHANGE,Aarav,Pune,aarav@example.com,2026-01-01
null,DELETE,null,null,null,null


**Step 4: Remove unchanged records**

In [0]:
changes_df = changes_df.filter(
    col("operation") != "NO_CHANGE"
)

**Step 5: Create a temporary view**

In [0]:
changes_df.createOrReplaceTempView("customer_changes")

**Step 6: Apply changes using MERGE INTO**

Now comes the actual CDC operation.

In [0]:
%sql
MERGE INTO cdc_lab.customers_batch_current AS target

USING customer_changes AS source

ON target.customer_id = source.customer_id

WHEN MATCHED AND source.operation = 'DELETE'
THEN DELETE

WHEN MATCHED AND source.operation = 'UPDATE'
THEN UPDATE SET
    target.customer_name = source.customer_name,
    target.city = source.city,
    target.email = source.email,
    target.updated_at = source.updated_at

WHEN NOT MATCHED AND source.operation = 'INSERT'
THEN INSERT (
    customer_id,
    customer_name,
    city,
    email,
    updated_at
)
VALUES (
    source.customer_id,
    source.customer_name,
    source.city,
    source.email,
    source.updated_at
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,2,0,1


**Step 7: Validate the result**

In [0]:
%sql
SELECT *
FROM cdc_lab.customers_batch_current
ORDER BY customer_id;

customer_id,customer_name,city,email,updated_at
101,Aarav,Pune,aarav@example.com,2026-01-01
102,Diya,Thane,diya@example.com,2026-02-01
103,Kabir,Nashik,kabir@example.com,2026-01-03
104,Meera,Nagpur,meera@example.com,2026-01-04
105,Ishaan,Pune,ishaan.k@example.com,2026-02-02
106,Sara,Delhi,sara@example.com,2026-01-06
107,Vivaan,Mumbai,vivaan@example.com,2026-01-07
108,Anaya,Nanded,anaya@example.com,2026-01-08
109,Rohan,Aurangabad,rohan@example.com,2026-02-03


**We implemented CDC without receiving individual change events from the source database.**

### Method 2: CDC using Delta MERGE INTO with a CDC event feed

We don't need to compare two complete snapshots anymore.

We simply process the change records.

**Step 1: Read CDC events**

In [0]:
df_cdc = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CDC_EVENTS)
)

display(df_cdc)

sequence_num,customer_id,operation,customer_name,city,email,event_time
1,101,UPDATE,Aarav,Pune,aarav@example.com,2026-02-01T09:00:00.000Z
2,102,UPDATE,Diya,Thane,diya@example.com,2026-02-01T09:05:00.000Z
3,109,INSERT,Rohan,Aurangabad,rohan@example.com,2026-02-01T09:10:00.000Z
4,103,DELETE,null,null,null,2026-02-01T09:15:00.000Z
5,105,UPDATE,Ishaan,Pune,ishaan.k@example.com,2026-02-01T09:20:00.000Z
6,110,INSERT,Tara,Kolhapur,tara@example.com,2026-02-01T09:25:00.000Z
7,102,UPDATE,Diya,Navi Mumbai,diya@example.com,2026-02-01T09:30:00.000Z
8,104,DELETE,null,null,null,2026-02-01T09:35:00.000Z


**Step 2: Understand the operation column**

In most CDC systems, you receive operation indicators. Some source systems use different codes, such as c, u, and d. You should standardize those codes before processing.

**Step 3: Create the target table**

We'll create a target table that also stores the last processed sequence number.

In [0]:
from pyspark.sql.functions import lit

df_target = (
    df_initial
    .withColumn("last_sequence_num", lit(0).cast("long"))
    .withColumn("is_deleted", lit(False))
)

df_target.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cdc_lab.customers_event_current")

In [0]:
display(df_target)

customer_id,customer_name,city,email,updated_at,last_sequence_num,is_deleted
101,Aarav,Pune,aarav@example.com,2026-01-01,0,false
102,Diya,Mumbai,diya@example.com,2026-01-02,0,false
103,Kabir,Nashik,kabir@example.com,2026-01-03,0,false
104,Meera,Nagpur,meera@example.com,2026-01-04,0,false
105,Ishaan,Pune,ishaan@example.com,2026-01-05,0,false
106,Sara,Delhi,sara@example.com,2026-01-06,0,false
107,Vivaan,Mumbai,vivaan@example.com,2026-01-07,0,false
108,Anaya,Nanded,anaya@example.com,2026-01-08,0,false


**Step 4: Deduplicate incoming events**

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc, col, lit

window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(desc("sequence_num"))
)

df_latest = (
    df_cdc
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
display(df_latest)

sequence_num,customer_id,operation,customer_name,city,email,event_time
1,101,UPDATE,Aarav,Pune,aarav@example.com,2026-02-01T09:00:00.000Z
7,102,UPDATE,Diya,Navi Mumbai,diya@example.com,2026-02-01T09:30:00.000Z
4,103,DELETE,null,null,null,2026-02-01T09:15:00.000Z
8,104,DELETE,null,null,null,2026-02-01T09:35:00.000Z
5,105,UPDATE,Ishaan,Pune,ishaan.k@example.com,2026-02-01T09:20:00.000Z
3,109,INSERT,Rohan,Aurangabad,rohan@example.com,2026-02-01T09:10:00.000Z
6,110,INSERT,Tara,Kolhapur,tara@example.com,2026-02-01T09:25:00.000Z


**Step 5: Create a temporary view**

In [0]:
df_latest.createOrReplaceTempView("latest_customer_changes")

**Step 6: Apply CDC using MERGE**

In [0]:
%sql
MERGE INTO cdc_lab.customers_event_current AS target

USING latest_customer_changes AS source

ON target.customer_id = source.customer_id

WHEN MATCHED
AND source.sequence_num > target.last_sequence_num

THEN UPDATE SET

    target.customer_name = source.customer_name,
    target.city = source.city,
    target.email = source.email,

    target.last_sequence_num = source.sequence_num,

    target.is_deleted =
        CASE
            WHEN source.operation = 'DELETE'
            THEN TRUE
            ELSE FALSE
        END

WHEN NOT MATCHED

THEN INSERT (
    customer_id,
    customer_name,
    city,
    email,
    last_sequence_num,
    is_deleted
)

VALUES (
    source.customer_id,
    source.customer_name,
    source.city,
    source.email,
    source.sequence_num,
    source.operation = 'DELETE'
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
7,5,0,2


**Step 7: Query the target table**

In [0]:
%sql
SELECT *
FROM cdc_lab.customers_event_current
ORDER BY customer_id;

customer_id,customer_name,city,email,updated_at,last_sequence_num,is_deleted
101,Aarav,Pune,aarav@example.com,2026-01-01,1,false
102,Diya,Navi Mumbai,diya@example.com,2026-01-02,7,false
103,null,null,null,2026-01-03,4,true
104,null,null,null,2026-01-04,8,true
105,Ishaan,Pune,ishaan.k@example.com,2026-01-05,5,false
106,Sara,Delhi,sara@example.com,2026-01-06,0,false
107,Vivaan,Mumbai,vivaan@example.com,2026-01-07,0,false
108,Anaya,Nanded,anaya@example.com,2026-01-08,0,false
109,Rohan,Aurangabad,rohan@example.com,null,3,false
110,Tara,Kolhapur,tara@example.com,null,6,false


To get only active customers:

In [0]:
%sql
SELECT *
FROM cdc_lab.customers_event_current
WHERE is_deleted = FALSE;

customer_id,customer_name,city,email,updated_at,last_sequence_num,is_deleted
106,Sara,Delhi,sara@example.com,2026-01-06,0,false
107,Vivaan,Mumbai,vivaan@example.com,2026-01-07,0,false
108,Anaya,Nanded,anaya@example.com,2026-01-08,0,false
101,Aarav,Pune,aarav@example.com,2026-01-01,1,false
102,Diya,Navi Mumbai,diya@example.com,2026-01-02,7,false
105,Ishaan,Pune,ishaan.k@example.com,2026-01-05,5,false
109,Rohan,Aurangabad,rohan@example.com,null,3,false
110,Tara,Kolhapur,tara@example.com,null,6,false


### Method 3: CDC using PySpark DataFrame operations

We can use PySpark DataFrames and Delta Lake APIs.

The logic is the same:
1. Identify matching records using the primary key.
2. Update matching records.
3. Insert new records.
4. Handle deletions.
5. Maintain event ordering.

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "cdc_lab.customers_event_current"
)

(
    target.alias("t")
    .merge(
        df_latest.alias("s"),
        "t.customer_id = s.customer_id"
    )
    .whenMatchedUpdate(
        condition="s.sequence_num > t.last_sequence_num",
        set={
            "customer_name": "s.customer_name",
            "city": "s.city",
            "email": "s.email",
            "last_sequence_num": "s.sequence_num",
            "is_deleted": "s.operation = 'DELETE'"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "s.customer_id",
            "customer_name": "s.customer_name",
            "city": "s.city",
            "email": "s.email",
            "last_sequence_num": "s.sequence_num",
            "is_deleted": "s.operation = 'DELETE'"
        }
    )
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Part B – Streaming CDC Implementation

### Method 4: Structured Streaming + foreachBatch + MERGE

Delta Lake's MERGE INTO is a batch operation. We can use foreachBatch() to apply that operation to each incoming streaming micro-batch.

**Step 1: Create a streaming input folder**

Inside your Volume, create a folder called:
```text
cdc_files/
    stream_inbox/
        customers_cdc_events.csv
```

In [0]:
STREAM_PATH = f"{BASE_PATH}/stream_inbox"

**Step 2: Create the initial target table**

In [0]:
from pyspark.sql.functions import lit

df_stream_target = (
    df_initial
    .withColumn(
        "last_sequence_num",
        lit(0).cast("long")
    )
    .withColumn(
        "is_deleted",
        lit(False)
    )
)

df_stream_target.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cdc_lab.customers_stream_current")

**Step 3: Read streaming CDC events using Auto Loader**

In [0]:
df_stream = (
    spark.readStream
    .format("cloudFiles")  #This is Auto Loader
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option(
        "cloudFiles.schemaLocation",
        f"{BASE_PATH}/_schemas/customers"
    )
    .load(STREAM_PATH)
)

```text
| Component           | Purpose          |
|---------------------|------------------|
| `readStream`        | Reads incoming files incrementally |
| `cloudFiles`        | Activates Auto Loader |
| `cloudFiles.format` | Defines source file format |
| `header`            | Reads CSV headers |
| `schemaLocation`    | Stores inferred schema information |
| `load()`            | Defines the input directory |
```

Unlike spark.read, which reads available files as a batch, spark.readStream establishes a streaming source.

**Step 4: Create the CDC processing function**

In [0]:
def process_cdc_batch(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    from pyspark.sql.window import Window
    from pyspark.sql.functions import row_number, desc, col

    # Deduplicate events within this micro-batch

    window_spec = (
        Window
        .partitionBy("customer_id")
        .orderBy(desc("sequence_num"))
    )

    latest_df = (
        batch_df
        .withColumn(
            "rn",
            row_number().over(window_spec)
        )
        .filter(col("rn") == 1)
        .drop("rn")
    )

    # Create temporary view

    latest_df.createOrReplaceTempView("stream_cdc_changes")

    # Apply CDC using Delta MERGE

    spark.sql("""
        MERGE INTO cdc_lab.customers_stream_current AS target

        USING stream_cdc_changes AS source

        ON target.customer_id = source.customer_id

        WHEN MATCHED
        AND source.sequence_num > target.last_sequence_num

        THEN UPDATE SET

            target.customer_name = source.customer_name,
            target.city = source.city,
            target.email = source.email,

            target.last_sequence_num = source.sequence_num,

            target.is_deleted =
                CASE
                    WHEN source.operation = 'DELETE'
                    THEN TRUE
                    ELSE FALSE
                END

        WHEN NOT MATCHED

        THEN INSERT (
            customer_id,
            customer_name,
            city,
            email,
            last_sequence_num,
            is_deleted
        )

        VALUES (
            source.customer_id,
            source.customer_name,
            source.city,
            source.email,
            source.sequence_num,
            source.operation = 'DELETE'
        )
    """)

    print(f"Processed micro-batch: {batch_id}")

Let's understand the processing logic:

1. A new micro-batch arrives.
2. We identify duplicate customer events within that micro-batch.
3. We retain the latest event per customer.
4. We create a temporary view.
5. We execute Delta MERGE.
6. We update existing records or insert new records.
7. We retain deletion information through is_deleted.

Let's understand the processing logic:
1. A new micro-batch arrives.
2. We identify duplicate customer events within that micro-batch.
3. We retain the latest event per customer.
4. We create a temporary view.
5. We execute Delta MERGE.
6. We update existing records or insert new records.
7. We retain deletion information through is_deleted.

**Step 5: Start the streaming query**

In [0]:
query = (
    df_stream.writeStream
    .foreachBatch(process_cdc_batch)
    .option(
        "checkpointLocation",
        f"{BASE_PATH}/_checkpoints/customers_cdc"
    )
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

**Step 6: Validate the streaming result**

In [0]:
%sql
SELECT *
FROM cdc_lab.customers_stream_current
ORDER BY customer_id;

customer_id,customer_name,city,email,updated_at,last_sequence_num,is_deleted
101,Aarav,Pune,aarav@example.com,2026-01-01,1,false
102,Diya,Navi Mumbai,diya@example.com,2026-01-02,7,false
103,null,null,null,2026-01-03,4,true
104,null,null,null,2026-01-04,8,true
105,Ishaan,Pune,ishaan.k@example.com,2026-01-05,5,false
106,Sara,Delhi,sara@example.com,2026-01-06,0,false
107,Vivaan,Mumbai,vivaan@example.com,2026-01-07,0,false
108,Anaya,Nanded,anaya@example.com,2026-01-08,0,false
109,Rohan,Aurangabad,rohan@example.com,null,3,false
110,Tara,Kolhapur,tara@example.com,null,6,false


**Step 7: Test incremental ingestion**

Now let's test something interesting.
1. Add another CDC CSV file to stream_inbox.
2. Give it a different filename.
3. Run the streaming notebook again using the same checkpoint.
4. Observe the target table.
Auto Loader uses the checkpoint and its file tracking state to avoid unnecessarily processing previously discovered files.

### Method 5: Auto Loader + Delta Lake + CDC

You might wonder: Didn't we already use Auto Loader?

Yes. But now let's understand the architecture from a production perspective.

Auto Loader is not itself a CDC engine.

It detects and incrementally ingests new files. The CDC processing logic is implemented downstream.

Consider this architecture:

**Source CDC files**

INSERT / UPDATE / DELETE

**Auto Loader**

Incremental file ingestion and schema tracking

**Bronze – Raw CDC**

Preserve incoming event records

**Silver – CDC processing**

Deduplication + sequence validation + MERGE

**Gold – Current customer data**

Business-ready customer records

### Part C – Advanced CDC Implementations

### Method 6: Databricks AUTO CDC (formerly APPLY CHANGES)

In modern Databricks, we have Lakeflow Spark Declarative Pipelines AUTO CDC.

Previously, the API was called APPLY CHANGES.

Why is this important?

Because Databricks manages much of the complex CDC processing automatically:
- Event ordering.
- Deduplication.
- Inserts.
- Updates.
- Deletes.
- SCD Type 1.
- SCD Type 2.
- Handling out-of-order records.
Instead of writing your own MERGE statements, you declare how changes should be applied.

```text
| Feature           | Manual MERGE   | AUTO CDC |
|-------------------|----------------|--------|
| Insert/update     | You implement  | Managed |
| Delete            | You implement  | Managed |
| Event sequencing  | You implement  | Managed |
| Deduplication     | You implement  | Managed |
| SCD Type 1        | Manual logic   | Built in |
| SCD Type 2        | More complex logic | Built in |
| Pipeline orchestration | Separate      | Integrated |
```